# Pre-Training Phase 1: Diverse Data

**Paper §2.3** — trains on 23.5 T tokens from a broad, diverse mixture including
web crawl, code, math, Wikipedia, academic text, multilingual text, and SFT-style
synthetic data.  The full Warmup-Stable-Decay (WSD) schedule is applied.

### Notebook contents
1. Environment setup
2. Imports & hyperparameters
3. Tokenizer
4. Dataset helpers
5. Learning-rate schedule & optimizer
6. Model
7. Loss & training step
8. Checkpointing
9. Training loop
10. Evaluation


## 0. Environment Setup

Detects Colab / Kaggle, installs packages, and adds the repo to `sys.path`.

In [ ]:
import os, sys

IN_COLAB  = False
IN_KAGGLE = os.path.exists("/kaggle/working") and os.path.exists("/kaggle/input")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print(f"Colab: {IN_COLAB} | Kaggle: {IN_KAGGLE}")


In [ ]:
if IN_COLAB or IN_KAGGLE:
    # Set ACCELERATOR to "cuda12" (GPU) or "tpu" (Colab TPU only).
    ACCELERATOR = "cuda12"
    import subprocess
    subprocess.run(
        ["pip", "install", "-q",
         f"jax[{ACCELERATOR}]", "flax", "optax", "orbax-checkpoint",
         "datasets", "transformers", "matplotlib"],
        check=True,
    )


In [ ]:
import pathlib

REPO_URL = "https://github.com/wisnunugroho21/nugie-jax-nemotron.git"


def _detect_repo_root() -> pathlib.Path:
    """Detect local repo root by searching upward for key project files."""
    cwd = pathlib.Path.cwd()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "training_shared.py").exists() and (candidate / "nemotron.py").exists():
            return candidate
    return cwd


if IN_COLAB:
    REPO_DIR = pathlib.Path("/content/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
elif IN_KAGGLE:
    REPO_DIR = pathlib.Path("/kaggle/working/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    REPO_DIR = _detect_repo_root()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("REPO_DIR:", REPO_DIR)
print(f"NUM_DEVICES={NUM_DEVICES}  (batch is sharded {BATCH_SIZE // NUM_DEVICES} samples/device)")


## 1. Imports & Hyperparameters

In [ ]:
import math
import pathlib
import sys

import jax
import jax.numpy as jnp
import numpy as np
import optax
import orbax.checkpoint as ocp
from datasets import load_dataset
from flax import nnx
from transformers import AutoTokenizer

from moe import SparseMoE
from nemotron import NemotronConfig, NemotronNanoBlock
from training_shared import (
    VOCAB_SIZE, CHUNK_SIZE,
    PRETRAIN_SEQ_LEN, PRETRAIN_BATCH, PHASE1_STEPS,
    PRETRAIN_PEAK_LR, PRETRAIN_MIN_LR,
    PRETRAIN_WARMUP_STEPS, PRETRAIN_STABLE_STEPS, PRETRAIN_DECAY_STEPS,
    PRETRAIN_WD, PRETRAIN_B1, PRETRAIN_B2,
    PRETRAIN_CKPT_DIR, PRETRAIN_CKPT_EVERY, PRETRAIN_VAL_STEPS,
    build_model, collect_moe_layers, make_decayed_lr_optimizer,
    load_pretrain_data, make_batches, pretrain_step, update_moe_biases,
    evaluate_pretrain, make_checkpoint_manager, save_checkpoint,
    NUM_DEVICES,
)

print("JAX devices:", jax.devices())


In [ ]:
# ── Hyperparameters (override here for quick experiments) ──────────────────────
SEQ_LEN      = PRETRAIN_SEQ_LEN   # 256
BATCH_SIZE   = PRETRAIN_BATCH     # 2
TRAIN_STEPS  = PHASE1_STEPS       # 5 000 (reduce for a quick smoke-test)
CKPT_DIR     = PRETRAIN_CKPT_DIR  # "./checkpoints_pretrain"

print(f"SEQ_LEN={SEQ_LEN} | BATCH={BATCH_SIZE} | STEPS={TRAIN_STEPS}")
print(f"WSD schedule: warmup={PRETRAIN_WARMUP_STEPS}, stable={PRETRAIN_STABLE_STEPS}, decay={PRETRAIN_DECAY_STEPS}")
print(f"NUM_DEVICES={NUM_DEVICES}  ({BATCH_SIZE // NUM_DEVICES} samples/device)")


## 2. Tokenizer

In [ ]:
print("Loading Nemotron tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained("nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size : {tokenizer.vocab_size}")
print(f"EOS token  : {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")


## 3. Dataset

Stream and pack texts from `HuggingFaceFW/fineweb-edu` into fixed-length chunks.

In [ ]:
# Phase 1 uses a large, diverse mixture; here we use a small FineWeb-Edu sample
# as a stand-in.  Increase max_samples for real training.
train_chunks = load_pretrain_data(
    split="train", max_samples=200, seq_len=SEQ_LEN, tokenizer=tokenizer, skip=0
)
val_chunks = load_pretrain_data(
    split="train", max_samples=50,  seq_len=SEQ_LEN, tokenizer=tokenizer, skip=200
)

print(f"Train chunks : {len(train_chunks)}  shape={train_chunks.shape}")
print(f"Val   chunks : {len(val_chunks)}   shape={val_chunks.shape}")


## 4. Model

In [ ]:
print("Building model ...")
model = build_model(seed=0)
moe_layers = collect_moe_layers(model)
print(f"MoE layers : {len(moe_layers)}")


## 5. Optimizer (WSD schedule)

In [ ]:
tx = make_decayed_lr_optimizer(
    peak_lr=PRETRAIN_PEAK_LR,
    min_lr=PRETRAIN_MIN_LR,
    warmup_steps=PRETRAIN_WARMUP_STEPS,
    stable_steps=PRETRAIN_STABLE_STEPS,
    decay_steps=PRETRAIN_DECAY_STEPS,
    weight_decay=PRETRAIN_WD,
    b1=PRETRAIN_B1,
    b2=PRETRAIN_B2,
)
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)
print("Optimizer created (AdamW + gradient clip + WSD schedule).")


## 6. Checkpoint Manager

In [ ]:
ckpt_mgr = make_checkpoint_manager(CKPT_DIR)
print(f"Checkpoints → {CKPT_DIR}  (every {PRETRAIN_CKPT_EVERY} steps)")


## 7. Training Loop

Phase 1 of the WSD schedule: warmup → stable → cosine decay over `PHASE1_STEPS` steps.
MoE bias update runs outside the gradient tape after every step.

In [ ]:
print(f"\n=== Pre-Training Phase 1: Diverse Data ===")
print(f"Training for {TRAIN_STEPS} steps (batch={BATCH_SIZE}, seq_len={SEQ_LEN}) ...")
print("(First step triggers JAX JIT compilation — it will be slow.)\n")

loss_history: list[float] = []
step = 0
batch_iter = iter(make_batches(train_chunks, BATCH_SIZE))

while step < TRAIN_STEPS:
    try:
        batch_np = next(batch_iter)
    except StopIteration:
        batch_iter = iter(make_batches(train_chunks, BATCH_SIZE))
        batch_np = next(batch_iter)

    batch = batch_np  # already sharded across all devices by make_batches
    loss  = pretrain_step(model, optimizer, batch)
    update_moe_biases(moe_layers)
    step += 1
    loss_history.append(float(loss))

    if step % 100 == 0:
        val_loss, ppl = evaluate_pretrain(model, val_chunks, PRETRAIN_VAL_STEPS, moe_layers)
        print(f"  Step {step:5d} | train_loss={float(loss):.4f} | "
              f"val_loss={val_loss:.4f} | ppl={ppl:.1f}")

    if step % PRETRAIN_CKPT_EVERY == 0:
        save_checkpoint(ckpt_mgr, model, step)

save_checkpoint(ckpt_mgr, model, step)
print("\nPhase 1 complete.")


## 8. Loss Curve

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4))
    plt.plot(range(1, len(loss_history) + 1), loss_history)
    plt.xlabel("Step"); plt.ylabel("Loss"); plt.title("Phase 1 Training Loss")
    plt.grid(True); plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed — skipping plot.")


## 9. Final Evaluation

In [ ]:
val_loss, ppl = evaluate_pretrain(model, val_chunks, PRETRAIN_VAL_STEPS, moe_layers)
print(f"Final val_loss={val_loss:.4f}  |  perplexity={ppl:.1f}")
